In [8]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import numpy as np
import os

In [39]:

# Configure TensorFlow logging and CPU usage
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Reduce TensorFlow logging
os.environ['OMP_NUM_THREADS'] = '4'  # Limit CPU threads
tf.config.threading.set_intra_op_parallelism_threads(4)
tf.config.threading.set_inter_op_parallelism_threads(4)

# Paths
train_dir = r"C:\Users\lotte\Downloads\merged_dataset\train"
test_dir = r"C:\Users\lotte\Downloads\merged_dataset\test"

# Parameters
img_height, img_width = 224, 224
batch_size = 32
epochs = 15
fine_tune_epochs = 10  # Extended for selective fine-tuning
num_classes = None  # Will be determined from train_generator

# Data Generators with Enhanced Augmentation
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,  # Increased rotation
    zoom_range=0.15,  # Increased zoom
    horizontal_flip=True,
    brightness_range=[0.3, 1.7],  # Wider brightness variation
    channel_shift_range=100.0,  # Stronger RGB shift for color diversity
    shear_range=0.3,  # Increased shear
    fill_mode='nearest'
)
test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='sparse'
)
val_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='sparse'
)

num_classes = len(train_generator.class_indices)

# Custom Attention Layer
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)
        self.conv = layers.Conv2D(1, (1, 1), activation='sigmoid', padding='same')

    def call(self, inputs):
        attention = self.conv(inputs)
        return layers.Multiply()([inputs, attention])

    def get_config(self):
        config = super(AttentionLayer, self).get_config()
        return config

# Load Pre-trained MobileNetV2
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(img_height, img_width, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Freeze base for initial training

# Build Model
model = models.Sequential([
    base_model,
    AttentionLayer(),  # Use custom attention layer
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

# Compile Model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Early Stopping
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Train
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=epochs,
    callbacks=[early_stop]
)

# Selective Fine-Tuning
base_model.trainable = True
for layer in base_model.layers[:100]:  # Freeze first 100 layers
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # Lower learning rate
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Fine-Tune
model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=fine_tune_epochs,
    callbacks=[early_stop]
)
# After fine-tuning
  # Saves as a directory

Found 46092 images belonging to 26 classes.
Found 11923 images belonging to 26 classes.
Epoch 1/15
1441/1441 ━━━━━━━━━━━━━━━━━━━━ 5052s 3s/step - accuracy: 0.3296 - loss: 4.3040 - val_accuracy: 0.8809 - val_loss: 1.7819
Epoch 2/15
1441/1441 ━━━━━━━━━━━━━━━━━━━━ 3290s 2s/step - accuracy: 0.7614 - loss: 1.9641 - val_accuracy: 0.9343 - val_loss: 1.1396
Epoch 3/15
1441/1441 ━━━━━━━━━━━━━━━━━━━━ 3251s 2s/step - accuracy: 0.8283 - loss: 1.3723 - val_accuracy: 0.9526 - val_loss: 0.8319
Epoch 4/15
1441/1441 ━━━━━━━━━━━━━━━━━━━━ 3214s 2s/step - accuracy: 0.8581 - loss: 1.0743 - val_accuracy: 0.9569 - val_loss: 0.6640
Epoch 5/15
1441/1441 ━━━━━━━━━━━━━━━━━━━━ 3221s 2s/step - accuracy: 0.8722 - loss: 0.9048 - val_accuracy: 0.9603 - val_loss: 0.5635
Epoch 6/15
1441/1441 ━━━━━━━━━━━━━━━━━━━━ 3345s 2s/step - accuracy: 0.8841 - loss: 0.7960 - val_accuracy: 0.9628 - val_loss: 0.5067
Epoch 7/15
1441/1441 ━━━━━━━━━━━━━━━━━━━━ 3434s 2s/step - accuracy: 0.8921 - loss: 0.7184 - val_accuracy: 0.9637 - val_l

In [40]:
model.save('isl.h5')  # New Keras V3 format (recommended)
  # Legacy H5 format

In [53]:
pip install mediapipe

   ---------------------------------------- 0.0/51.0 MB ? eta -:--:--
   -- ------------------------------------- 3.1/51.0 MB 18.5 MB/s eta 0:00:03
   ----- ---------------------------------- 7.3/51.0 MB 18.9 MB/s eta 0:00:03
   --------- ------------------------------ 12.1/51.0 MB 20.4 MB/s eta 0:00:02
   ------------ --------------------------- 16.5/51.0 MB 20.8 MB/s eta 0:00:02
   ---------------- ----------------------- 21.5/51.0 MB 21.2 MB/s eta 0:00:02
   -------------------- ------------------- 26.0/51.0 MB 21.4 MB/s eta 0:00:02
   ----------------------- ---------------- 29.9/51.0 MB 21.1 MB/s eta 0:00:02
   -------------------------- ------------- 33.8/51.0 MB 20.4 MB/s eta 0:00:01
   ----------------------------- ---------- 37.0/51.0 MB 20.2 MB/s eta 0:00:01
   ----------------------------- ---------- 37.5/51.0 MB 19.9 MB/s eta 0:00:01
   -------------------------------- ------- 41.2/51.0 MB 17.9 MB/s eta 0:00:01
   ---------------------------------- ----- 44.0/51.0 MB 17.6 M

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.


In [55]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image

class ISLErrorDetector:
    def __init__(self, model_path):
        self.model = tf.keras.models.load_model(model_path)
        self.class_names = ['A', 'B', 'C', 'D', 'E']  # Replace with your actual class names
        
        # Define feedback rules for each letter
        self.feedback_rules = {
            'A': [
                "Make sure your thumb is touching your index finger",
                "Other fingers should be fully curled into your palm"
            ],
            'B': [
                "All fingers should be straight and close together",
                "Thumb should be across the palm, not sticking out"
            ],
            # Add more letters as needed
        }
    
    def load_and_preprocess(self, img_path):
        """Load and preprocess image using only TensorFlow"""
        img = image.load_img(img_path, target_size=(224, 224))
        img_array = image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)
        return img_array / 255.0
    
    def analyze_image(self, img_path):
        """Analyze image and provide feedback"""
        try:
            # Load and predict
            img_array = self.load_and_preprocess(img_path)
            prediction = self.model.predict(img_array)
            class_idx = np.argmax(prediction[0])
            confidence = np.max(prediction[0]) * 100
            letter = self.class_names[class_idx]
            
            # Get feedback
            feedback = self.feedback_rules.get(letter, ["No specific feedback available"])
            
            # Display results
            self.show_results(img_path, letter, confidence, feedback)
            
            return {
                'letter': letter,
                'confidence': confidence,
                'feedback': feedback
            }
            
        except Exception as e:
            print(f"Error processing image: {e}")
            return None
    
    def show_results(self, img_path, letter, confidence, feedback):
        """Display results using matplotlib"""
        img = image.load_img(img_path)
        
        plt.figure(figsize=(10, 8))
        plt.imshow(img)
        plt.title(f"Predicted: {letter} ({confidence:.1f}% confidence)", fontsize=16)
        
        if feedback:
            feedback_text = "Feedback:\n" + "\n".join(f"- {item}" for item in feedback)
            plt.figtext(0.5, 0.05, feedback_text, 
                       ha="center", 
                       fontsize=12,
                       bbox={"facecolor":"orange", "alpha":0.3, "pad":5})
        
        plt.axis('off')
        plt.tight_layout()
        plt.show()

# Usage Example:
detector = ISLErrorDetector("your_model.keras")
result = detector.analyze_image("test_image.jpg")

ValueError: File not found: filepath=your_model.keras. Please ensure the file is an accessible `.keras` zip file.

In [ ]:
def _generate_feedback(self, predicted_class, landmarks):
    feedback = []
    
    # Example rules for letter 'A'
    if predicted_class == 'A':
        thumb_tip = landmarks[4]
        index_tip = landmarks[8]
        
        # Check thumb-index distance (should be touching)
        if self._calc_distance(thumb_tip, index_tip) > 0.1:
            feedback.append("Bring thumb and index finger closer together")
            
        # Check palm orientation
        if landmarks[0].y < landmarks[9].y:
            feedback.append("Rotate palm more forward")
    
    # Add rules for other letters...
    
    return feedback if feedback else "Good form!"

In [ ]:
def visualize_corrections(image_path, feedback):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(img)
    
    # Add annotation arrows for corrections
    for i, comment in enumerate(feedback):
        ax.annotate(comment, 
                   xy=(0.5, 0.95 - i*0.05),
                   xycoords='axes fraction',
                   color='red',
                   fontsize=12)
    
    plt.axis('off')
    plt.show()